# Bab 5 — Research Methodology and Experimental Protocol

Notebook pendamping Bab 5 Level 7. Jalankan sel dari atas ke bawah, dan **jalankan skrip atau notebook Bab 2 dan Bab 4 lebih dulu**. Bagian A sampai G mengikuti halaman bab. Bagian **Praktik** berisi kegiatan yang tidak ada di halaman: menulis protokol mini untuk pertanyaanmu sendiri, membuat amendment, dan menghitung ulang jumlah eksekusi bila desain berubah.

Notebook ini **tidak menghitung metrik hasil eksperimen**. Hasil eksperimen ada di Bab 6.

## Learning Objectives

Setelah mengerjakan notebook ini, kamu mampu:

- Menulis protokol mini yang lengkap untuk satu pertanyaan dan memvalidasinya.
- Membuat amendment yang tidak menimpa protokol terkunci.
- Menghitung ulang jumlah eksekusi dari desain, bukan dari angka yang diketik.

## Langkah 0: Pindah ke folder utama proyek

In [1]:
import os
import sys
from pathlib import Path

if not Path("data").exists() and not Path("examples").exists():
    os.chdir("../..")

ROOT = Path.cwd()
# alat bantu bersama Level 7 (examples/level-7/_research.py)
sys.path.insert(0, str(Path("examples/level-7").resolve()))

## Bagian A: Siapkan alat dan baca bab sebelumnya

In [2]:
import copy
import hashlib
import json
import platform
import time
from importlib import metadata

import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

import _research as rs

RESULTS = rs.RESULTS_DIR
problem = json.loads((RESULTS / "02_research_problem.json").read_text(encoding="utf-8"))
card = json.loads((RESULTS / "04_dataset_card.json").read_text(encoding="utf-8"))
X_all, diagnosis = rs.load_wdbc()
print("Pertanyaan:", list(problem["pertanyaan"]), "| sidik jari data final:", card["sidik_jari_data_final"])

Pertanyaan: ['RQ1', 'RQ2', 'RQ3'] | sidik jari data final: 6b202a2072f9


## Bagian B: Menulis protokol

In [3]:
protocol = {
    "versi": "1.0",
    "judul_kerja": problem["judul_kerja"],
    "dataset": {
        "nama": card["nama"],
        "sidik_jari": card["sidik_jari_data_final"],
        "jumlah_sampel": card["jumlah_sampel_final"],
        "jumlah_fitur": 30,
        "label": "diagnosis: hanya untuk validasi eksternal, tidak dipakai melatih atau menyetel",
    },
    "pertanyaan": problem["pertanyaan"],
    "analisis": {
        "RQ1": {
            "faktor": "seed dan sampel data",
            "algoritma": ["KMeans"],
            "ulangan": {"seed": 20, "subsampel": 50, "fraksi_subsampel": 0.8},
            "metrik": ["ARI antar-seed", "ARI pengelompokan subsampel vs data penuh"],
        },
        "RQ2": {
            "faktor": "perlakuan skala fitur",
            "level": ["StandardScaler (baseline)", "tanpa skala", "MinMaxScaler"],
            "algoritma": ["KMeans"],
            "ulangan": {"seed": 20},
            "metrik": ["ARI vs baseline"],
        },
        "RQ3": {
            "faktor": "algoritma",
            "algoritma": ["KMeans", "Agglomerative", "GMM", "DBSCAN"],
            "metrik": ["ARI vs diagnosis", "NMI vs diagnosis"],
            "metrik_internal": ["Silhouette", "Davies-Bouldin", "Calinski-Harabasz"],
        },
        "sensitivitas_K": {
            "faktor": "jumlah kelompok K",
            "level": [2, 3, 4, 5, 6],
            "algoritma": ["KMeans", "Agglomerative", "GMM"],
            "metrik": ["Silhouette"],
        },
    },
    "algoritma": {
        "KMeans": {"n_clusters": 2, "n_init": 10, "disetel": False},
        "Agglomerative": {"n_clusters": 2, "linkage": "ward", "disetel": False},
        "GMM": {"n_components": 2, "covariance_type": "full", "n_init": 5, "disetel": False},
        "DBSCAN": {
            "min_samples": [5, 10],
            "eps_kuantil_jarak_tetangga": [0.80, 0.85, 0.90, 0.95],
            "disetel": True,
            "aturan_seleksi": "Silhouette tertinggi di antara konfigurasi dengan minimal 2 kelompok dan noise paling banyak 10 persen",
            "jika_tak_ada_yang_valid": "dilaporkan 'tidak ada konfigurasi valid'; aturan tidak dilonggarkan",
        },
    },
    "praproses": {
        "baseline": "StandardScaler pada ke-30 fitur",
        "outlier": "tidak dibuang (keputusan Bab 4)",
        "seleksi_fitur": "tidak ada pada analisis utama",
    },
    "kebijakan_hyperparameter": {
        "K_analisis_utama": 2,
        "justifikasi_K": "RQ3 membandingkan dengan diagnosis dua kelas; K = 2 adalah asumsi desain, bukan hasil pemilihan; sensitivitas K dilaporkan terpisah",
        "label_untuk_tuning": False,
    },
    "pembanding_adil": [
        "data dan praproses sama untuk semua algoritma",
        "metrik internal dihitung di ruang baseline yang sama",
        "label tidak dipakai untuk memilih konfigurasi",
        "keterbatasan: hanya DBSCAN yang disetel, jadi upaya penyetelan tidak setara",
    ],
    "kondisi_terkontrol": {"random_state_dasar": 42, "catatan": "versi pustaka dicatat terpisah di 05_environment.json"},
    "aturan_keputusan": {
        "ambang_praktis_sama_ARI": 0.95,
        "catatan": "ambang ditetapkan sebelum eksperimen dan tidak digeser setelah melihat hasil",
    },
    "pengungkapan": [
        "Bab 1 menjalankan demonstrasi K-Means pada data yang sama dan menampilkan ARI terhadap diagnosis sebelum protokol ini ditulis",
        "protokol ini karena itu tidak sepenuhnya buta terhadap data; K = 2 dipengaruhi jumlah kelas diagnosis",
    ],
    "amendments": [],
}
print("Bagian protokol:", list(protocol))
print("Algoritma:", list(protocol["algoritma"]))

Bagian protokol: ['versi', 'judul_kerja', 'dataset', 'pertanyaan', 'analisis', 'algoritma', 'praproses', 'kebijakan_hyperparameter', 'pembanding_adil', 'kondisi_terkontrol', 'aturan_keputusan', 'pengungkapan', 'amendments']
Algoritma: ['KMeans', 'Agglomerative', 'GMM', 'DBSCAN']


## Bagian C: Memvalidasi protokol

In [4]:
def validate(p):
    """Mengembalikan daftar masalah. Daftar kosong berarti protokol lolos pemeriksaan bentuk (bukan kebenaran ilmiah)."""
    issues = []
    for key in ["versi", "dataset", "pertanyaan", "analisis", "algoritma", "praproses", "kebijakan_hyperparameter", "pembanding_adil"]:
        if key not in p:
            issues.append(f"bagian '{key}' tidak ada")
    for rq in p.get("pertanyaan", {}):
        if rq not in p.get("analisis", {}):
            issues.append(f"{rq} tidak punya rencana analisis")
        elif not p["analisis"][rq].get("metrik"):
            issues.append(f"{rq} tidak punya metrik")
    if p.get("kebijakan_hyperparameter", {}).get("label_untuk_tuning") is not False:
        issues.append("label tidak boleh dipakai untuk menyetel")
    if not p.get("dataset", {}).get("sidik_jari"):
        issues.append("sidik jari data tidak tercatat")
    if p.get("analisis", {}).get("RQ1", {}).get("ulangan", {}).get("seed", 0) < 10:
        issues.append("RQ1 butuh minimal 10 seed")
    defined = set(p.get("algoritma", {}))
    for name, plan in p.get("analisis", {}).items():
        for algo in plan.get("algoritma", []):
            if algo not in defined:
                issues.append(f"{name} memakai algoritma '{algo}' yang tidak didefinisikan")
    for item in p.get("amendments", []):
        if not item.get("alasan"):
            issues.append("ada amendment tanpa alasan")
    return issues


print("Protokol asli, masalah:", validate(protocol))

broken = copy.deepcopy(protocol)
del broken["analisis"]["RQ2"]
broken["kebijakan_hyperparameter"]["label_untuk_tuning"] = True
broken["dataset"]["sidik_jari"] = ""
broken["analisis"]["RQ3"]["algoritma"].append("SpectralClustering")
for issue in validate(broken):
    print("  - protokol rusak:", issue)

Protokol asli, masalah: []
  - protokol rusak: RQ2 tidak punya rencana analisis
  - protokol rusak: label tidak boleh dipakai untuk menyetel
  - protokol rusak: sidik jari data tidak tercatat
  - protokol rusak: RQ3 memakai algoritma 'SpectralClustering' yang tidak didefinisikan


## Bagian D: Menghitung jumlah eksekusi

In [5]:
a = protocol["analisis"]
dbscan = protocol["algoritma"]["DBSCAN"]
runs = {
    "RQ1 seed": a["RQ1"]["ulangan"]["seed"],
    "RQ1 subsampel": a["RQ1"]["ulangan"]["subsampel"],
    "RQ2 level x seed": len(a["RQ2"]["level"]) * a["RQ2"]["ulangan"]["seed"],
    "RQ3 algoritma tak disetel": sum(not v["disetel"] for v in protocol["algoritma"].values()),
    "RQ3 grid DBSCAN": len(dbscan["min_samples"]) * len(dbscan["eps_kuantil_jarak_tetangga"]),
    "Sensitivitas K": len(a["sensitivitas_K"]["level"]) * len(a["sensitivitas_K"]["algoritma"]),
}
grid = pd.DataFrame({"jumlah_eksekusi": runs})
grid.loc["TOTAL"] = grid["jumlah_eksekusi"].sum()
print(grid.to_string())

                           jumlah_eksekusi
RQ1 seed                                20
RQ1 subsampel                           50
RQ2 level x seed                        60
RQ3 algoritma tak disetel                3
RQ3 grid DBSCAN                          8
Sensitivitas K                          15
TOTAL                                  156


## Bagian E: Smoke test tanpa metrik berbasis label

In [6]:
# Smoke test: apakah setiap konfigurasi bisa dijalankan? TIDAK memakai label dan TIDAK menghitung metrik hasil.
X_scaled = StandardScaler().fit_transform(X_all)
cfg = protocol["algoritma"]
models = {
    "KMeans": KMeans(n_clusters=cfg["KMeans"]["n_clusters"], n_init=cfg["KMeans"]["n_init"], random_state=42),
    "Agglomerative": AgglomerativeClustering(n_clusters=cfg["Agglomerative"]["n_clusters"], linkage=cfg["Agglomerative"]["linkage"]),
    "GMM": GaussianMixture(n_components=cfg["GMM"]["n_components"], covariance_type=cfg["GMM"]["covariance_type"], n_init=cfg["GMM"]["n_init"], random_state=42),
}
k = dbscan["min_samples"][0]
distances = NearestNeighbors(n_neighbors=k).fit(X_scaled).kneighbors(X_scaled)[0][:, -1]
eps = float(np.quantile(distances, dbscan["eps_kuantil_jarak_tetangga"][0]))
models["DBSCAN"] = DBSCAN(eps=eps, min_samples=k)

smoke_rows = []
for name, model in models.items():
    start = time.perf_counter()
    labels = model.fit_predict(X_scaled)
    elapsed = time.perf_counter() - start
    n_clusters = len(set(labels) - {-1})
    smoke_rows.append({"algoritma": name, "berjalan": True, "kelompok": n_clusters, "noise": int((labels == -1).sum()), "detik": round(elapsed, 3)})
smoke = pd.DataFrame(smoke_rows).set_index("algoritma")
print(smoke.to_string())
print("Smoke test tidak menghitung ARI/NMI; hasilnya tidak dilaporkan sebagai temuan.")

               berjalan  kelompok  noise  detik
algoritma                                      
KMeans             True         2      0  0.061
Agglomerative      True         2      0  0.066
GMM                True         2      0  0.168
DBSCAN             True         1     75  0.012
Smoke test tidak menghitung ARI/NMI; hasilnya tidak dilaporkan sebagai temuan.


## Bagian F: Mengunci protokol dengan hash

In [7]:
def protocol_hash(p):
    text = json.dumps(p, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


locked = protocol_hash(protocol)
RESULTS.mkdir(exist_ok=True)
(RESULTS / "05_protocol.json").write_text(json.dumps(protocol, indent=2, ensure_ascii=False), encoding="utf-8")
(RESULTS / "05_protocol.sha256").write_text(locked + "\n", encoding="utf-8")
environment = {"python": platform.python_version(), "os": platform.system()}
for package in ("numpy", "pandas", "scikit-learn"):
    environment[package] = metadata.version(package)
(RESULTS / "05_environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")

reloaded = json.loads((RESULTS / "05_protocol.json").read_text(encoding="utf-8"))
print("Hash protokol    :", locked[:16], "...")
print("Hash setelah baca:", "sama" if protocol_hash(reloaded) == locked else "BERBEDA")
tampered = copy.deepcopy(protocol)
tampered["kebijakan_hyperparameter"]["K_analisis_utama"] = 3
print("Hash setelah K diubah menjadi 3:", "sama" if protocol_hash(tampered) == locked else "BERBEDA")

Hash protokol    : c65be407e00ab39b ...
Hash setelah baca: sama
Hash setelah K diubah menjadi 3: BERBEDA


## Bagian G: Mencatat perubahan protokol

In [8]:
def amend(p, perubahan, alasan, tanggal):
    """Perubahan setelah protokol dikunci TIDAK menimpa isi lama: dicatat sebagai amendment berbalas alasan."""
    updated = copy.deepcopy(p)
    updated["amendments"].append({"tanggal": tanggal, "perubahan": perubahan, "alasan": alasan})
    return updated


example = amend(protocol, "contoh: menambah 10 seed pada RQ1", "contoh latihan saja, tidak dipakai", "2026-11-10")
print("Amendment pada protokol asli:", len(protocol["amendments"]))
print("Amendment pada salinan contoh:", len(example["amendments"]))
print("Hash salinan contoh berbeda dari protokol terkunci:", protocol_hash(example) != locked)
print("Masalah pada salinan contoh:", validate(example))

Amendment pada protokol asli: 0
Amendment pada salinan contoh: 1
Hash salinan contoh berbeda dari protokol terkunci: True
Masalah pada salinan contoh: []


## Praktik 1: Protokol mini untuk pertanyaanmu

Ganti `ISI` dengan rancanganmu sendiri untuk satu pertanyaan (boleh pertanyaan baru). `validate_mini` memeriksa bahwa bagian wajib terisi dan bahwa label tidak dipakai menyetel. Lolos pemeriksaan hanya berarti **lengkap**, bukan **bagus**.

In [9]:
mini = {
    "pertanyaan": "ISI",
    "faktor": "ISI",
    "level": ["ISI"],
    "terkontrol": ["ISI"],
    "metrik": ["ISI"],
    "ulangan": 0,
    "aturan_keputusan": "ISI",
    "label_untuk_tuning": False,
}

def validate_mini(m):
    issues = []
    for key in ["pertanyaan", "faktor", "aturan_keputusan"]:
        if str(m.get(key, "")).strip() in ("", "ISI"):
            issues.append(f"{key} belum diisi")
    for key in ["level", "terkontrol", "metrik"]:
        if not m.get(key) or "ISI" in m[key]:
            issues.append(f"{key} belum diisi")
    if m.get("ulangan", 0) < 1:
        issues.append("ulangan minimal 1")
    if m.get("label_untuk_tuning") is not False:
        issues.append("label tidak boleh dipakai menyetel")
    return issues

print("Masalah protokol mini:", validate_mini(mini))

Masalah protokol mini: ['pertanyaan belum diisi', 'faktor belum diisi', 'aturan_keputusan belum diisi', 'level belum diisi', 'terkontrol belum diisi', 'metrik belum diisi', 'ulangan minimal 1']


## Praktik 2: Buat amendment

Bayangkan kamu menemukan bahwa 20 seed untuk RQ1 terlalu sedikit. Ubah angkanya, isi alasannya, lalu lihat bahwa protokol terkunci tidak berubah, dan versi baru memiliki hash berbeda. Selama `alasan` masih kosong, validasi akan menolaknya.

In [10]:
new_seed_count = 30
alasan = ""   # tulis alasan yang jujur, misalnya "kebisingan antar-seed belum cukup terukur dengan 20 seed"

v2 = amend(protocol, f"RQ1: seed 20 -> {new_seed_count}", alasan, "2026-11-10")
v2["analisis"]["RQ1"]["ulangan"]["seed"] = new_seed_count
print("Seed pada protokol terkunci:", protocol["analisis"]["RQ1"]["ulangan"]["seed"])
print("Seed pada versi baru       :", v2["analisis"]["RQ1"]["ulangan"]["seed"])
print("Hash berbeda dari yang terkunci:", protocol_hash(v2) != locked)
print("Masalah versi baru:", validate(v2))

Seed pada protokol terkunci: 20
Seed pada versi baru       : 30
Hash berbeda dari yang terkunci: True
Masalah versi baru: ['ada amendment tanpa alasan']


## Praktik 3: Hitung ulang jumlah eksekusi

Ubah desain pada salinan (jumlah seed, level skala, atau grid DBSCAN) lalu hitung ulang total tanpa mengetik angka. Tujuannya melihat berapa mahal sebuah keputusan desain sebelum kamu menjalankannya.

In [11]:
import math

trial = copy.deepcopy(protocol)
trial["analisis"]["RQ2"]["ulangan"]["seed"] = 30
trial["analisis"]["RQ2"]["level"].append("RobustScaler")

def total_runs(p):
    a, d = p["analisis"], p["algoritma"]["DBSCAN"]
    return (a["RQ1"]["ulangan"]["seed"] + a["RQ1"]["ulangan"]["subsampel"]
            + len(a["RQ2"]["level"]) * a["RQ2"]["ulangan"]["seed"]
            + sum(not v["disetel"] for v in p["algoritma"].values())
            + len(d["min_samples"]) * len(d["eps_kuantil_jarak_tetangga"])
            + len(a["sensitivitas_K"]["level"]) * len(a["sensitivitas_K"]["algoritma"]))

print("Total protokol terkunci :", total_runs(protocol))
print("Total desain percobaan  :", total_runs(trial))

Total protokol terkunci : 156
Total desain percobaan  : 216


## Latihan

1. Isi `mini` untuk satu pertanyaanmu sampai `validate_mini` mengembalikan daftar kosong. Lalu tulis satu kelemahan rancanganmu yang tidak bisa ditangkap `validate_mini`.
2. Isi `alasan` pada Praktik 2 dengan alasan yang jujur. Apakah amendment ini perlu dilakukan **sebelum** atau **sesudah** melihat hasil? Mengapa waktunya penting?
3. Pada Praktik 3, tambahkan satu level skala lagi dan naikkan seed RQ1. Berapa totalnya sekarang?
4. Ubah `label_untuk_tuning` menjadi `True` pada salinan protokol dan jalankan `validate`. Apa yang dilaporkan?

## Ringkasan

- Protokol ditulis sebelum eksperimen dan memuat faktor, level, ulangan, metrik, aturan keputusan, dan pembanding yang adil.
- Label tidak boleh dipakai memilih konfigurasi; pengungkapan hal yang sudah dilihat ditulis di protokol.
- Hash mengunci desain, dan perubahan sah dicatat sebagai amendment dengan alasan, bukan menimpa.
- Jumlah eksekusi dihitung dari desain, sehingga biaya setiap keputusan terlihat sebelum eksperimen berjalan.